In [ ]:
import os
import torch
import scipy.io as sio
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Dataset, Data, InMemoryDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, global_mean_pool
import torch.optim as optim
from sklearn.model_selection import GroupShuffleSplit  # Use group-based splitting to avoid patient-level leakage.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from torch_geometric.utils import dense_to_sparse  # Convert dense adjacency matrices to sparse format.

# ------------------------------------------------------------------------------
# ImprovedBrainNetworkDataset: Corrected to persist custom patient_id and use weights_only=False.
# ------------------------------------------------------------------------------
class ImprovedBrainNetworkDataset(InMemoryDataset):
    def __init__(self, root, response_path, transform=None, pre_transform=None):
        self.response_path = response_path
        super().__init__(root, transform, pre_transform)
        # Explicitly set weights_only=False to ensure all attributes are loaded.
        self.data, self.slices = torch.load(self.processed_paths[0], weights_only=False)
        
        # Load patient IDs separately.
        self.patient_ids = torch.load(self.processed_paths[1], weights_only=False)
    
    @property
    def processed_file_names(self):
        return ['processed_graphs.pt', 'patient_ids.pt']
    
    def process(self):
        data_list = []
        patient_ids = []  # Separate list for patient IDs.
        responses = pd.read_csv(self.response_path)
        responses_dict = dict(zip(responses['patient_id'], responses['label']))
        
        mat_files = [os.path.join(self.raw_dir, f) for f in os.listdir(self.raw_dir) if f.endswith('.mat')]
        
        patient_files = {}
        for mat_path in mat_files:          
            # File naming convention: MXXXXV.mat (XXXX = patient ID, V = visit number)
            # Adapt this parsing to match your dataset's naming convention
            patient_id = int(os.path.basename(mat_path).split('M')[1][:4])
            visit_num = int(os.path.basename(mat_path).split('M')[1][4])
            
            if patient_id not in patient_files:
                patient_files[patient_id] = []
            patient_files[patient_id].append((visit_num, mat_path))
        
        for patient_id, visits in patient_files.items():
            visits.sort()
            visit_features = []
            for i, (visit_num, mat_path) in enumerate(visits):
                mat_data = sio.loadmat(mat_path)
                dti_r = mat_data['structural_connectivity']['r'][0][0]
                rest_r = mat_data['functional_connectivity']['r'][0][0]
                combined_adj = (dti_r + rest_r) / 2

                if i > 0:
                    change = combined_adj - previous_adj
                    visit_features.append(change)
                previous_adj = combined_adj
            
            if visit_features:
                avg_change = np.mean(visit_features, axis=0)
            else:
                avg_change = np.zeros_like(combined_adj)
            
            adj_matrix = torch.tensor(avg_change, dtype=torch.float32)
            edge_index, edge_attr = dense_to_sparse(adj_matrix)
            
            label = responses_dict.get(patient_id, 0)
            data = Data(
                x=adj_matrix,
                edge_index=edge_index,
                edge_attr=edge_attr,
                y=torch.tensor([label], dtype=torch.long)
            )
            data_list.append(data)
            patient_ids.append(patient_id)  # Store patient ID separately
        
        data, slices = self.collate(data_list)
        torch.save((data, slices), self.processed_paths[0])
        torch.save(patient_ids, self.processed_paths[1])  # Save patient IDs separately

    def get(self, idx):
        data = super().get(idx)
        data.patient_id = self.patient_ids[idx]  # Attach patient ID to data object.
        return data


# ------------------------------------------------------------------------------
# ImprovedBrainNetworkGNN: Removed unused edge_attr in GAT layers.
# ------------------------------------------------------------------------------
class ImprovedBrainNetworkGNN(nn.Module):
    def __init__(self, num_features, hidden_channels=64, dropout_rate=0.3):
        super().__init__()
        
        # Graph Attention Layers.
        self.gat1 = GATConv(num_features, hidden_channels, heads=4, concat=True)
        self.gat2 = GATConv(hidden_channels * 4, hidden_channels // 2, heads=2, concat=False)  # Removed edge_attr parameter.
        
        # Batch Normalization.
        self.bn1 = nn.BatchNorm1d(hidden_channels * 4)
        self.bn2 = nn.BatchNorm1d(hidden_channels // 2)
        
        # Dropout.
        self.dropout = nn.Dropout(dropout_rate)
        
        # Final Classification Layer.
        self.fc = nn.Linear(hidden_channels // 2, 1)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = self.gat1(x, edge_index)  # Forward pass without passing edge_attr.
        x = self.bn1(x)
        x = F.leaky_relu(x)
        x = self.dropout(x)

        x = self.gat2(x, edge_index)
        x = self.bn2(x)
        x = F.leaky_relu(x)

        # Global pooling to form graph-level representation.
        x = global_mean_pool(x, batch)
        return self.fc(x)


# ------------------------------------------------------------------------------
# train_and_evaluate: Uses group-aware splitting to avoid leakage across patient data.
# ------------------------------------------------------------------------------
def train_and_evaluate(dataset, test_size=0.2, batch_size=32, epochs=50, learning_rate=0.001):
    # Extract patient_ids directly from the dataset.
    patient_ids = dataset.patient_ids
    indices = np.arange(len(dataset))
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=42)  # GroupShuffleSplit for group-aware partition.
    train_idx, test_idx = next(gss.split(indices, groups=patient_ids))
    
    train_dataset = dataset[train_idx]
    test_dataset = dataset[test_idx]
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = ImprovedBrainNetworkGNN(num_features=dataset[0].x.shape[1]).to(device)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-5)
    criterion = nn.BCEWithLogitsLoss()

    # Training Loop.
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch in train_loader:
            batch.to(device)  # Move batch to device (GPU/CPU).
            optimizer.zero_grad()

            output = model(batch)
            loss = criterion(output.view(-1), batch.y.float())  # Reshape output and labels for loss calculation.
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Gradient clipping to stabilize training.
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")

    # Evaluation.
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in test_loader:
            batch.to(device)  # Move batch to device (GPU/CPU).
            output = torch.sigmoid(model(batch)).cpu()
            preds = (output > 0.5).numpy().flatten()
            labels = batch.y.cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(labels)

    print("\nTest Set Performance:")
    print("Accuracy:", accuracy_score(all_labels, all_preds))
    print("Precision:", precision_score(all_labels, all_preds))
    print("Recall:", recall_score(all_labels, all_preds))
    print("F1 Score:", f1_score(all_labels, all_preds))

    cm = confusion_matrix(all_labels, all_preds)
    print("\nConfusion Matrix:\n", cm)


# ------------------------------------------------------------------------------
# Main: Initializes dataset and triggers training and evaluation.
# ------------------------------------------------------------------------------
def main():
    # Update these paths to point to your local data directory
    dataset_path =  r'path/to/your/mat_files_root_directory'
    response_path = r'path/to/your/labels.csv'

    # Ensure that dataset_path contains a 'raw' subdirectory with the .mat files.
    dataset = ImprovedBrainNetworkDataset(
        root=dataset_path,
        response_path=response_path
    )

    train_and_evaluate(dataset)

if __name__ == "__main__":
    main()


